# VN Stock Screener
**Sang loc co phieu Viet Nam theo 2 tang loc (Market Filter + Financial Strength)**

Nguon du lieu: `vnstock` (VCI source) | 6 nganh x 5 ma = 30 co phieu

In [26]:
# Cell 1: Cai thu vien
!pip install vnstock openpyxl pandas numpy


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [27]:
# Cell 2: Import thu vien va cau hinh hien thi
import time
import sys
import warnings
import collections
import ast

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 160)
pd.set_option('display.float_format', '{:.2f}'.format)

print('Import OK')

Import OK


In [28]:
# Cell 3: Danh sach ma co phieu theo nganh
# 6 nganh x 10 ma = 60 ma tong cong

SECTOR_STOCKS = {
    'Xay dung ha tang': [
        'HBC', 'CTD', 'VCG', 'FCN', 'PC1',
        'C4G', 'HHV', 'LCG', 'CII', 'DIG',
    ],
    'Khu cong nghiep & logistics': [
        'KBC', 'SZC', 'IDC', 'GMD', 'HAH',
        'PHP', 'DVP', 'STG', 'TMS', 'BCG',
    ],
    'San xuat cong nghe cao': [
        'VCS', 'ELC', 'SAM', 'MWG', 'STK',
        'TNG', 'MSH', 'GIL', 'TCM', 'QNS',
    ],
    'Cong nghe & dich vu so': [
        'FPT', 'CMG', 'VGI', 'ITD', 'BTT',
        'SGT', 'POT', 'TST', 'VNP', 'CMT',
    ],
    'Nang luong tai tao': [
        'REE', 'GEG', 'TV2', 'VSH', 'HDG',
        'POW', 'NT2', 'SHP', 'BWE', 'PGV',
    ],
    'Y te & bao hiem': [
        'DHG', 'IMP', 'DMC', 'BVH', 'PTI',
        'TRA', 'DBD', 'MED', 'TNH', 'DVN',
    ],
}

ALL_SYMBOLS = []
SYMBOL_TO_SECTOR = {}
for sector, symbols in SECTOR_STOCKS.items():
    for s in symbols:
        ALL_SYMBOLS.append(s)
        SYMBOL_TO_SECTOR[s] = sector

TOTAL = len(ALL_SYMBOLS)
print('Tong so ma:', TOTAL)
for sector, symbols in SECTOR_STOCKS.items():
    print('  {} ({}): {}'.format(sector, len(symbols), ', '.join(symbols)))


Tong so ma: 60
  Xay dung ha tang (10): HBC, CTD, VCG, FCN, PC1, C4G, HHV, LCG, CII, DIG
  Khu cong nghiep & logistics (10): KBC, SZC, IDC, GMD, HAH, PHP, DVP, STG, TMS, BCG
  San xuat cong nghe cao (10): VCS, ELC, SAM, MWG, STK, TNG, MSH, GIL, TCM, QNS
  Cong nghe & dich vu so (10): FPT, CMG, VGI, ITD, BTT, SGT, POT, TST, VNP, CMT
  Nang luong tai tao (10): REE, GEG, TV2, VSH, HDG, POW, NT2, SHP, BWE, PGV
  Y te & bao hiem (10): DHG, IMP, DMC, BVH, PTI, TRA, DBD, MED, TNH, DVN


In [29]:
# Cell 4: Cau hinh bo loc
#
# Cu phap rule:
#   'ten_cot': ('toan_tu', he_so, loai)
#   loai = 'rel' -> so voi trung binh nganh (he_so x mean)
#   loai = 'abs' -> nguong tuyet doi
#
# Vi du:
#   'roe': ('>=', 1.0, 'rel')  -> ROE >= 1.0 x trung binh nganh
#   'market_cap': ('>=', 500, 'abs')  -> von hoa >= 500 ty

# Tang 1: Market Filter
TIER1_RULES = [
    ('market_cap', '>=', 500,  'abs'),   # von hoa >= 500 ty
    ('pe',         '>=', 0,    'abs'),   # loai PE am
    ('pe',         '<=', 1.0,  'rel'),   # PE <= trung binh nganh
    ('pb',         '<=', 1.0,  'rel'),   # PB <= trung binh nganh
    ('ev_ebitda',  '<=', 1.0,  'rel'),   # EV/EBITDA <= trung binh nganh
]

# Tang 2: Financial Strength
TIER2_RULES = [
    ('roe',              '>=', 1.0,  'rel'),
    ('roic',             '>=', 1.0,  'rel'),
    ('gross_margin',     '>=', 1.0,  'rel'),
    ('ebit_margin',      '>=', 1.0,  'rel'),
    ('debt_equity',      '<=', 1.0,  'rel'),
    ('interest_coverage','>=', 1.0,  'rel'),
]

# Override cho nganh dac thu (xay dung, nang luong su dung nhieu no hon)
SECTOR_OVERRIDES = {
    'Xay dung ha tang': {
        'debt_equity':        ('<=', 1.3, 'rel'),
        'interest_coverage':  ('>=', 0.8, 'rel'),
    },
    'Nang luong tai tao': {
        'debt_equity':        ('<=', 1.3, 'rel'),
        'interest_coverage':  ('>=', 0.8, 'rel'),
    },
}

# Trong so tinh diem xep hang
SCORE_WEIGHTS = {
    'roe':              0.20,
    'roic':             0.15,
    'gross_margin':     0.10,
    'ebit_margin':      0.10,
    'pe':               0.15,
    'pb':               0.10,
    'debt_equity':      0.10,
    'interest_coverage':0.10,
}
# Chi so nghich (thap = tot): dung 1/ratio
INVERSE_METRICS = {'pe', 'pb', 'ev_ebitda', 'debt_equity'}

print('Cau hinh bo loc OK')

Cau hinh bo loc OK


In [30]:
# Cell 5: Ham tien ich, COLUMN_MAP, RateLimiter

COLUMN_MAP = {
    'P/E':                    'pe',
    'P/B':                    'pb',
    'EV/EBITDA':              'ev_ebitda',
    'P/S':                    'ps',
    'Von hoa thi truong':     'market_cap',
    'Von hoa':                'market_cap',
    'ROE (%)':                'roe',
    'ROA (%)':                'roa',
    'ROIC (%)':               'roic',
    'Bien loi nhuan gop (%)': 'gross_margin',
    'Bien EBIT (%)':          'ebit_margin',
    'Bien loi nhuan rong (%)':'net_margin',
    'No/VCSH':                'debt_equity',
    'No dai han/VCSH':        'lt_debt_equity',
    'Kha nang thanh toan lai vay': 'interest_coverage',
    'Tang truong doanh thu (%)': 'revenue_growth',
    'Tang truong loi nhuan rong (%)': 'net_income_growth',
    'Tang truong EPS (%)':    'eps_growth',
    'Kha nang thanh toan hien hanh': 'current_ratio',
    'Kha nang thanh toan nhanh':     'quick_ratio',
}

def normalize_columns(df):
    new_cols = {}
    for col in df.columns:
        if isinstance(col, tuple):
            raw = col[-1]
        else:
            raw = str(col)
        parts = raw.split('\n')
        key = parts[-1].strip()
        mapped = COLUMN_MAP.get(key)
        if mapped is None:
            for k, v in COLUMN_MAP.items():
                if k.lower() in key.lower() or key.lower() in k.lower():
                    mapped = v
                    break
        new_cols[col] = mapped if mapped else (
            key.lower()
               .replace(' ', '_').replace('/', '_')
               .replace('(', '').replace(')', '').replace('%', 'pct')
        )
    return df.rename(columns=new_cols)


def get_latest_year(df):
    year_col = None
    for c in ['year', 'nam', 'period', 'ticker']:
        if c in df.columns:
            year_col = c
            break
    if year_col is None:
        return df.iloc[-1:].copy()
    try:
        df[year_col] = pd.to_numeric(df[year_col], errors='coerce')
        latest = df[year_col].max()
        return df[df[year_col] == latest].copy()
    except Exception:
        return df.iloc[-1:].copy()


# ---------------------------------------------------------------
# CHIEN LUOC TRANH RATE LIMIT
# ---------------------------------------------------------------
# INTER_REQUEST_DELAY = 10s: moi request cach nhau it nhat 10s
#   => toi da 6 req/phut, con du gap 3 lan so voi gioi han 20/phut
# RateLimiter chi la lop du phong them, hiem khi can den
# ---------------------------------------------------------------

INTER_REQUEST_DELAY = 7  # giay nghi bat buoc giua moi request


class RateLimiter:
    def __init__(self, safe_calls=18, window_secs=62):
        self.safe_calls = safe_calls
        self.window_secs = window_secs
        self.timestamps = collections.deque()

    def wait_if_needed(self):
        now = time.time()
        while self.timestamps and now - self.timestamps[0] > self.window_secs:
            self.timestamps.popleft()
        if len(self.timestamps) >= self.safe_calls:
            wait_sec = self.window_secs - (now - self.timestamps[0]) + 1
            if wait_sec > 0:
                print('    [RateLimiter] Dat {} req/window, nghi {}s...'.format(
                    self.safe_calls, int(wait_sec)))
                time.sleep(wait_sec)
        self.timestamps.append(time.time())


def fetch_ratio_with_retry(symbol, rate_limiter, max_retries=3):
    from vnstock import Finance
    for attempt in range(1, max_retries + 1):
        try:
            rate_limiter.wait_if_needed()
            time.sleep(INTER_REQUEST_DELAY)  # delay co dinh 10s/request
            fin = Finance(symbol=symbol, period='year', source='VCI')
            df = fin.ratio(lang='vi')
            if isinstance(df.columns, pd.MultiIndex):
                sep = '\n'
                df.columns = [
                    '{}{}{}'.format(a, sep, b) if b else a
                    for a, b in df.columns
                ]
            df = normalize_columns(df)
            df_latest = get_latest_year(df)
            year_val = 'N/A'
            for c in ['year', 'nam', 'period']:
                if c in df_latest.columns:
                    year_val = df_latest[c].values[0]
                    break
            if year_val == 'N/A' and not df_latest.empty:
                year_val = 'latest'
            return df_latest, year_val
        except Exception as e:
            err_msg = str(e).lower()
            is_rate = ('rate' in err_msg or 'limit' in err_msg
                       or '429' in err_msg or 'gioi han' in err_msg
                       or 'exceeded' in err_msg or 'toi da' in err_msg)
            if attempt < max_retries:
                wait_retry = 65 if is_rate else 5 * attempt
                tag = 'Rate limit' if is_rate else 'Loi'
                print('    [Retry {}/{}] {} | nghi {}s...'.format(
                    attempt, max_retries, tag, wait_retry))
                time.sleep(wait_retry)
            else:
                raise


print('Ham tien ich OK')
print('Rate limit strategy: {}s delay/request => toi da 6 req/phut'.format(
    INTER_REQUEST_DELAY))


Ham tien ich OK
Rate limit strategy: 7s delay/request => toi da 6 req/phut


In [31]:
# Cell 6: Fetch du lieu cho tat ca 60 ma
#
# Voi INTER_REQUEST_DELAY=10s: moi request cach nhau 10s
# => toi da 6 req/phut, khong bao gio cham gioi han 20/phut
# Tong thoi gian uoc tinh: 60 ma x ~12s = ~12 phut

BATCH_SIZE     = 19   # chi dung de hien thi label batch
COUNTDOWN_STEP = 5

rate_limiter = RateLimiter(safe_calls=18, window_secs=62)
records = []
errors  = []
start_total = time.time()

SEP1 = '-' * 60
SEP2 = '*' * 60

idx_global = 0

for sector, symbols in SECTOR_STOCKS.items():
    print(SEP1)
    print('Nganh: {}'.format(sector))
    print(SEP1)
    for sym in symbols:
        idx_global += 1
        batch_pos = (idx_global - 1) % BATCH_SIZE + 1
        elapsed   = int(time.time() - start_total)

        label = '[{:02d}/{}] {:<5} | batch: {}/{} | da chay: {}s'.format(
            idx_global, TOTAL, sym, batch_pos, BATCH_SIZE, elapsed
        )
        print('  {} ...'.format(label), end=' ', flush=True)

        try:
            df_sym, year_val = fetch_ratio_with_retry(sym, rate_limiter)
            df_sym['symbol'] = sym
            df_sym['sector'] = sector
            records.append(df_sym)
            print('OK (nam {})'.format(year_val))
        except Exception as e:
            errors.append((sym, str(e)))
            print('LOI: {}'.format(str(e)[:80]))

print()
print('HOAN THANH: {}/{} ma thanh cong, {} loi.'.format(
    len(records), TOTAL, len(errors)))
if errors:
    print('Cac ma bi loi:')
    for sym, err in errors:
        print('  {} -> {}'.format(sym, err))

if records:
    df_all = pd.concat(records, ignore_index=True, sort=False)
    meta_c = ['symbol', 'sector', 'year', 'nam', 'period']
    num_c = [c for c in df_all.columns if c not in meta_c]
    for c in num_c:
        df_all[c] = pd.to_numeric(df_all[c], errors='coerce')
    print('df_all shape:', df_all.shape)
    print('Cac cot:', list(df_all.columns))


------------------------------------------------------------
Nganh: Xay dung ha tang
------------------------------------------------------------
  [01/60] HBC   | batch: 1/19 | da chay: 0s ... OK (nam latest)
  [02/60] CTD   | batch: 2/19 | da chay: 8s ... OK (nam latest)
  [03/60] VCG   | batch: 3/19 | da chay: 16s ... OK (nam latest)
  [04/60] FCN   | batch: 4/19 | da chay: 25s ... 

KeyboardInterrupt: 

In [7]:
# Cell 7: Tinh trung binh nganh

META_COLS = ['symbol', 'sector', 'year', 'nam', 'period']
meta_present = [c for c in META_COLS if c in df_all.columns]
num_cols = [c for c in df_all.columns if c not in meta_present]

df_sector_mean = df_all.groupby('sector')[num_cols].mean()

# Hien thi cac chi so quan trong
display_cols = [c for c in ['pe', 'pb', 'ev_ebitda', 'roe', 'roic', 'gross_margin', 'ebit_margin', 'debt_equity', 'interest_coverage', 'market_cap'] if c in df_sector_mean.columns]

print('TRUNG BINH NGANH:')
print(SEP1)
display(df_sector_mean[display_cols].round(2))

TRUNG BINH NGANH:
------------------------------------------------------------


,pe,pb,ev_ebitda,roe,roic
sector,,,,,
Cong nghe & dich vu so,75.90,2.73,22.36,0.07,0.08
Khu cong nghiep & logistics,66.27,3.80,30.46,0.14,0.11
Nang luong tai tao,93.06,2.25,16.06,0.11,0.08
San xuat cong nghe cao,24.51,2.64,14.05,0.18,0.12
Xay dung ha tang,21.61,1.51,14.91,0.08,0.08
Y te & bao hiem,15.72,1.94,4.08,0.14,0.01


In [8]:
# Cell 8: Ham loc va chay bo loc 2 tang

def apply_rules(row, rules, sector_mean_row, overrides=None):
    """
    Ap dung danh sach rules cho 1 hang (row).
    rules: list of (col, op, he_so, loai)
    overrides: dict {col: (op, he_so, loai)} ghi de rule mac dinh
    Tra ve True neu tat ca rules deu pass.
    """
    # Tao dict rule mac dinh, override theo cot
    rule_dict = {}
    for col, op, he_so, loai in rules:
        rule_dict[col] = (op, he_so, loai)
    if overrides:
        for col, rule_tuple in overrides.items():
            rule_dict[col] = rule_tuple

    for col, (op, he_so, loai) in rule_dict.items():
        if col not in row.index:
            continue
        val = row[col]
        if pd.isna(val):
            return False  # Thieu du lieu -> khong pass

        if loai == 'rel':
            if col not in sector_mean_row.index or pd.isna(sector_mean_row[col]):
                continue
            threshold = he_so * sector_mean_row[col]
        else:  # abs
            threshold = he_so

        if op == '>=':
            if not (val >= threshold):
                return False
        elif op == '<=':
            if not (val <= threshold):
                return False
        elif op == '>':
            if not (val > threshold):
                return False
        elif op == '<':
            if not (val < threshold):
                return False
    return True


# Chay bo loc
results = []
filter_report = []

for sector, group_df in df_all.groupby('sector'):
    sector_mean_row = df_sector_mean.loc[sector] if sector in df_sector_mean.index else pd.Series(dtype=float)
    overrides = SECTOR_OVERRIDES.get(sector, {})

    # Tang 1
    mask_t1 = group_df.apply(
        lambda row: apply_rules(row, TIER1_RULES, sector_mean_row, {}),
        axis=1
    )
    df_t1 = group_df[mask_t1]

    # Tang 2 (chi ap dung cho ma da qua Tang 1)
    mask_t2 = df_t1.apply(
        lambda row: apply_rules(row, TIER2_RULES, sector_mean_row, overrides),
        axis=1
    )
    df_t2 = df_t1[mask_t2]

    results.append(df_t2)

    total_sector = len(group_df)
    pass_t1 = len(df_t1)
    pass_t2 = len(df_t2)
    pct = '{:.0f}%'.format(100 * pass_t2 / total_sector) if total_sector else '0%'

    filter_report.append({
        'Nganh': sector,
        'Tong ma': total_sector,
        'Pass Tang 1': pass_t1,
        'Pass Tang 2': pass_t2,
        'Pass Ca Hai': pass_t2,
        'Ty le': pct,
    })

df_passed = pd.concat(results, ignore_index=True) if results else pd.DataFrame()
df_filter_report = pd.DataFrame(filter_report)

print('BAO CAO LOC THEO NGANH:')
print(SEP1)
display(df_filter_report)
print()
print('Tong so ma qua 2 tang loc: {}'.format(len(df_passed)))

BAO CAO LOC THEO NGANH:
------------------------------------------------------------


,Nganh,Tong ma,Pass Tang 1,Pass Tang 2,Pass Ca Hai,Ty le
0,Cong nghe & dich vu so,10,5,2,2,20%
1,Khu cong nghiep & logistics,10,6,3,3,30%
2,Nang luong tai tao,10,3,0,0,0%
3,San xuat cong nghe cao,10,3,2,2,20%
4,Xay dung ha tang,10,4,3,3,30%
5,Y te & bao hiem,10,1,0,0,0%



Tong so ma qua 2 tang loc: 10


In [9]:
# Cell 9: Hien thi ket qua cac ma da qua loc

if df_passed.empty:
    print('Khong co ma nao qua ca 2 tang loc.')
else:
    print('CAC MA VUOT QUA 2 TANG LOC:')
    print(SEP1)

    show_cols_base = ['symbol', 'sector']
    show_cols_num = [c for c in ['pe', 'pb', 'ev_ebitda', 'roe', 'roic', 'gross_margin', 'ebit_margin', 'debt_equity', 'interest_coverage', 'market_cap'] if c in df_passed.columns]
    show_cols = show_cols_base + show_cols_num

    df_show = df_passed[show_cols].copy()

    # Format % cho cac chi so margin/sinh loi
    pct_cols = [c for c in ['roe', 'roic', 'gross_margin', 'ebit_margin', 'net_margin'] if c in df_show.columns]
    for c in pct_cols:
        df_show[c] = df_show[c].apply(lambda x: '{:.1f}%'.format(x) if pd.notna(x) else 'N/A')

    display(df_show.reset_index(drop=True))

CAC MA VUOT QUA 2 TANG LOC:
------------------------------------------------------------


,symbol,sector,pe,pb,ev_ebitda,roe,roic
0,BTT,Cong nghe & dich vu so,11.28,1.97,11.18,0.2%,0.1%
1,CMT,Cong nghe & dich vu so,19.98,1.48,6.52,0.1%,0.1%
2,PHP,Khu cong nghiep & logistics,9.41,1.88,4.95,0.2%,0.1%
3,DVP,Khu cong nghiep & logistics,11.11,3.04,7.58,0.3%,0.2%
4,STG,Khu cong nghiep & logistics,8.76,1.55,4.15,0.2%,0.2%
5,MWG,San xuat cong nghe cao,5.48,1.75,4.30,0.4%,0.3%
6,MSH,San xuat cong nghe cao,5.38,1.26,4.56,0.3%,0.1%
7,CTD,Xay dung ha tang,11.24,1.26,8.79,0.1%,0.1%
8,FCN,Xay dung ha tang,5.44,0.90,2.77,0.2%,0.2%
9,PC1,Xay dung ha tang,4.86,1.46,3.21,0.4%,0.3%


In [10]:
# Cell 10: So sanh tung ma voi trung binh nganh

compare_cols = [c for c in ['pe', 'pb', 'ev_ebitda', 'roe', 'roic', 'gross_margin', 'ebit_margin', 'debt_equity', 'interest_coverage'] if c in df_all.columns]

compare_records = []

for _, row in df_all.iterrows():
    sym = row.get('symbol', '')
    sec = row.get('sector', '')
    if sec not in df_sector_mean.index:
        continue
    mean_row = df_sector_mean.loc[sec]

    for col in compare_cols:
        if col not in row.index or col not in mean_row.index:
            continue
        val = row[col]
        mean_val = mean_row[col]
        if pd.isna(val) or pd.isna(mean_val) or mean_val == 0:
            continue
        ratio = val / mean_val
        # Tot hon / Kem hon theo loai chi so
        if col in INVERSE_METRICS:
            nhan_xet = 'Tot hon' if val <= mean_val else 'Kem hon'
        else:
            nhan_xet = 'Tot hon' if val >= mean_val else 'Kem hon'

        compare_records.append({
            'Symbol': sym,
            'Sector': sec,
            'Chi so': col,
            'Gia tri CP': round(val, 3),
            'TB Nganh': round(mean_val, 3),
            'Ty le (CP/TB)': round(ratio, 3),
            'Nhan xet': nhan_xet,
        })

df_compare = pd.DataFrame(compare_records)

print('BANG SO SANH CHI TIET VS TRUNG BINH NGANH (ma da qua loc):')
print(SEP1)
if not df_passed.empty:
    passed_symbols = df_passed['symbol'].tolist()
    display(df_compare[df_compare['Symbol'].isin(passed_symbols)].reset_index(drop=True))
else:
    print('Khong co ma nao qua loc.')

BANG SO SANH CHI TIET VS TRUNG BINH NGANH (ma da qua loc):
------------------------------------------------------------


,Symbol,Sector,Chi so,Gia tri CP,TB Nganh,Ty le (CP/TB),Nhan xet
0,CTD,Xay dung ha tang,pe,11.24,21.61,0.52,Tot hon
1,CTD,Xay dung ha tang,pb,1.25,1.51,0.83,Tot hon
2,CTD,Xay dung ha tang,ev_ebitda,8.79,14.91,0.59,Tot hon
3,CTD,Xay dung ha tang,roe,0.12,0.08,1.52,Tot hon
4,CTD,Xay dung ha tang,roic,0.10,0.08,1.27,Tot hon
5,FCN,Xay dung ha tang,pe,5.44,21.61,0.25,Tot hon
6,FCN,Xay dung ha tang,pb,0.90,1.51,0.60,Tot hon
7,FCN,Xay dung ha tang,ev_ebitda,2.77,14.91,0.19,Tot hon
8,FCN,Xay dung ha tang,roe,0.21,0.08,2.70,Tot hon
9,FCN,Xay dung ha tang,roic,0.17,0.08,2.06,Tot hon


In [11]:
# Cell 11: Xep hang theo diem tong hop (Score)

def compute_score(row, sector_mean_row, weights, inverse_metrics):
    """
    Tinh diem tong hop co trong so.
    ratio = cp_value / mean_nganh
    Chi so nghich (thap = tot): dung 1/ratio
    Score = sum(ratio * weight)
    """
    score = 0.0
    total_weight = 0.0
    for col, w in weights.items():
        if col not in row.index or col not in sector_mean_row.index:
            continue
        val = row[col]
        mean_val = sector_mean_row[col]
        if pd.isna(val) or pd.isna(mean_val) or mean_val == 0:
            continue
        ratio = val / mean_val
        if col in inverse_metrics:
            if ratio > 0:
                ratio = 1.0 / ratio
            else:
                continue
        score += ratio * w
        total_weight += w
    # Chuan hoa theo tong trong so thuc te
    if total_weight > 0:
        score = score / total_weight
    return round(score, 4)


# Tinh score cho tat ca ma da qua loc
if not df_passed.empty:
    scores = []
    for _, row in df_passed.iterrows():
        sec = row.get('sector', '')
        if sec in df_sector_mean.index:
            s = compute_score(row, df_sector_mean.loc[sec], SCORE_WEIGHTS, INVERSE_METRICS)
        else:
            s = 0.0
        scores.append(s)

    df_passed = df_passed.copy()
    df_passed['score'] = scores

    print('XEP HANG THEO DIEM TONG HOP (trong moi nganh):')
    print(SEP1)
    rank_cols = ['symbol', 'sector', 'score'] + [c for c in ['pe', 'pb', 'roe', 'roic', 'debt_equity'] if c in df_passed.columns]

    for sector, grp in df_passed.groupby('sector'):
        grp_sorted = grp.sort_values('score', ascending=False)
        print('\nNganh: {}'.format(sector))
        display(grp_sorted[rank_cols].reset_index(drop=True))
else:
    print('Khong co ma nao de xep hang.')

XEP HANG THEO DIEM TONG HOP (trong moi nganh):
------------------------------------------------------------

Nganh: Cong nghe & dich vu so


,symbol,sector,score,pe,pb,roe,roic
0,BTT,Cong nghe & dich vu so,3.15,11.28,1.97,0.17,0.14
1,CMT,Cong nghe & dich vu so,2.02,19.98,1.48,0.08,0.13



Nganh: Khu cong nghiep & logistics


,symbol,sector,score,pe,pb,roe,roic
0,STG,Khu cong nghiep & logistics,3.18,8.76,1.55,0.18,0.19
1,DVP,Khu cong nghiep & logistics,2.97,11.11,3.04,0.31,0.22
2,PHP,Khu cong nghiep & logistics,2.90,9.41,1.88,0.21,0.12



Nganh: San xuat cong nghe cao


,symbol,sector,score,pe,pb,roe,roic
0,MWG,San xuat cong nghe cao,2.70,5.48,1.75,0.41,0.26
1,MSH,San xuat cong nghe cao,2.24,5.38,1.26,0.25,0.13



Nganh: Xay dung ha tang


,symbol,sector,score,pe,pb,roe,roic
0,PC1,Xay dung ha tang,3.66,4.86,1.46,0.36,0.28
1,FCN,Xay dung ha tang,2.69,5.44,0.90,0.21,0.17
2,CTD,Xay dung ha tang,1.50,11.24,1.26,0.12,0.10


In [12]:
# Cell 12: Xuat Excel (5 sheet)

import openpyxl

OUTPUT_FILE = 'vn_stock_screener_result.xlsx'

with pd.ExcelWriter(OUTPUT_FILE, engine='openpyxl') as writer:

    # Sheet 1: Passed Stocks
    if not df_passed.empty:
        df_passed.to_excel(writer, sheet_name='Passed Stocks', index=False)
    else:
        pd.DataFrame({'Note': ['Khong co ma nao qua loc']}).to_excel(writer, sheet_name='Passed Stocks', index=False)

    # Sheet 2: All Raw Data
    df_all.to_excel(writer, sheet_name='All Raw Data', index=False)

    # Sheet 3: Sector Mean
    df_sector_mean.to_excel(writer, sheet_name='Sector Mean')

    # Sheet 4: Filter Report
    df_filter_report.to_excel(writer, sheet_name='Filter Report', index=False)

    # Sheet 5: Compare vs Mean
    df_compare.to_excel(writer, sheet_name='Compare vs Mean', index=False)

print('Da luu file:', OUTPUT_FILE)
print('Cac sheet: Passed Stocks | All Raw Data | Sector Mean | Filter Report | Compare vs Mean')

Da luu file: vn_stock_screener_result.xlsx
Cac sheet: Passed Stocks | All Raw Data | Sector Mean | Filter Report | Compare vs Mean


In [13]:
# Cell 13: Loc lai nhanh (khong can fetch lai du lieu)
# Chi can chinh TIER1_RULES_CUSTOM / TIER2_RULES_CUSTOM roi chay cell nay

# ============================
# Tuy chinh nguong o day
# ============================
TIER1_RULES_CUSTOM = [
    ('market_cap', '>=', 500,  'abs'),
    ('pe',         '>=', 0,    'abs'),
    ('pe',         '<=', 1.0,  'rel'),
    ('pb',         '<=', 1.0,  'rel'),
    ('ev_ebitda',  '<=', 1.0,  'rel'),
]

TIER2_RULES_CUSTOM = [
    ('roe',              '>=', 1.0,  'rel'),
    ('roic',             '>=', 1.0,  'rel'),
    ('gross_margin',     '>=', 1.0,  'rel'),
    ('ebit_margin',      '>=', 1.0,  'rel'),
    ('debt_equity',      '<=', 1.0,  'rel'),
    ('interest_coverage','>=', 1.0,  'rel'),
]
# ============================

results_custom = []
report_custom = []

for sector, group_df in df_all.groupby('sector'):
    sector_mean_row = df_sector_mean.loc[sector] if sector in df_sector_mean.index else pd.Series(dtype=float)
    overrides = SECTOR_OVERRIDES.get(sector, {})

    mask_t1 = group_df.apply(
        lambda row: apply_rules(row, TIER1_RULES_CUSTOM, sector_mean_row, {}),
        axis=1
    )
    df_t1 = group_df[mask_t1]

    mask_t2 = df_t1.apply(
        lambda row: apply_rules(row, TIER2_RULES_CUSTOM, sector_mean_row, overrides),
        axis=1
    )
    df_t2 = df_t1[mask_t2]
    results_custom.append(df_t2)

    total_sector = len(group_df)
    pass_t2 = len(df_t2)
    pct = '{:.0f}%'.format(100 * pass_t2 / total_sector) if total_sector else '0%'
    report_custom.append({
        'Nganh': sector,
        'Tong ma': total_sector,
        'Pass Tang 1': len(df_t1),
        'Pass Tang 2': pass_t2,
        'Ty le': pct,
    })

df_passed_custom = pd.concat(results_custom, ignore_index=True) if results_custom else pd.DataFrame()

print('KET QUA LOC LAI NHANH:')
print(SEP1)
display(pd.DataFrame(report_custom))
print()
print('Tong so ma qua loc:', len(df_passed_custom))

if not df_passed_custom.empty:
    show_c = ['symbol', 'sector'] + [c for c in ['pe', 'pb', 'roe', 'roic', 'debt_equity', 'market_cap'] if c in df_passed_custom.columns]
    display(df_passed_custom[show_c].reset_index(drop=True))

KET QUA LOC LAI NHANH:
------------------------------------------------------------


,Nganh,Tong ma,Pass Tang 1,Pass Tang 2,Ty le
0,Cong nghe & dich vu so,10,5,2,20%
1,Khu cong nghiep & logistics,10,6,3,30%
2,Nang luong tai tao,10,3,0,0%
3,San xuat cong nghe cao,10,3,2,20%
4,Xay dung ha tang,10,4,3,30%
5,Y te & bao hiem,10,1,0,0%



Tong so ma qua loc: 10


,symbol,sector,pe,pb,roe,roic
0,BTT,Cong nghe & dich vu so,11.28,1.97,0.17,0.14
1,CMT,Cong nghe & dich vu so,19.98,1.48,0.08,0.13
2,PHP,Khu cong nghiep & logistics,9.41,1.88,0.21,0.12
3,DVP,Khu cong nghiep & logistics,11.11,3.04,0.31,0.22
4,STG,Khu cong nghiep & logistics,8.76,1.55,0.18,0.19
5,MWG,San xuat cong nghe cao,5.48,1.75,0.41,0.26
6,MSH,San xuat cong nghe cao,5.38,1.26,0.25,0.13
7,CTD,Xay dung ha tang,11.24,1.26,0.12,0.10
8,FCN,Xay dung ha tang,5.44,0.90,0.21,0.17
9,PC1,Xay dung ha tang,4.86,1.46,0.36,0.28
